<a id="otimizacao-regressao-logistica"></a>

# TC3-60 — Otimização: Regressão Logística

**Responsabilidade sugerida:** Modelo linear e interpretável de referência.

Este notebook executa uma rodada intermediária e comparável de otimização. Ele usa exatamente o mesmo contrato, amostra, separação por grupos, métricas e orçamento de busca dos demais candidatos da Fase 6.

> Execute este notebook localmente. O conjunto de teste permanece reservado até que o grupo selecione um único modelo.


## 1. Protocolo comum

- `chave_escola_ano` é usada para impedir que alunos do mesmo agrupamento escola-ano apareçam simultaneamente em treino e validação;
- a chave **não é feature** e não identifica a escola real;
- treino e validação da Fase 4 compõem o universo de desenvolvimento;
- o split `teste` não é lido;
- a seleção prioriza ROC AUC e também registra average precision, balanced accuracy, F1, estabilidade e gap.


## Inicialização independente

Esta célula reproduz a configuração compartilhada do notebook principal. Ela prepara ambiente, caminhos e funções comuns sem alterar a lógica canônica desta etapa.


In [1]:
# =============================================================================
# CONFIGURAÇÕES PARA RODAR O NOTEBOOK
# Execute esta célula primeiro. Esta política não altera a metodologia do fluxo.
# =============================================================================
import os
import sys
import gc
import ctypes
import polars as pl
from datetime import datetime, timezone

# Opções do usuário
PERFIL_RECURSOS = "auto"       # auto | baixo | medio | normal | alto
LIMITE_RAM_PERCENTUAL = None    # None = automático; ou inteiro entre 50 e 90
MAX_THREADS = None              # None = automático; ou inteiro >= 1
USAR_GPU = os.getenv("TC3_USAR_GPU", "1").strip().lower() in {"1", "true", "sim", "yes"}                 # requer WSL2 + RAPIDS/cuML + kernel tc3-gpu

REUTILIZAR_ARTEFATOS = False  # False reconstrói; True aceita caches válidos
SALVAR_ARTEFATOS = True       # False usa somente área temporária da sessão


# Protocolo metodologico compartilhado por todas as fases e modelos.
SEED_GLOBAL = 42
EXECUTAR_SHAP = os.getenv("TC3_EXECUTAR_SHAP", "0").strip().lower() in {"1", "true", "sim", "yes"}
AMOSTRA_SHAP = max(20, int(os.getenv("TC3_AMOSTRA_SHAP", "100")))
AMOSTRA_BACKGROUND_SHAP = max(20, int(os.getenv("TC3_BACKGROUND_SHAP", "50")))
CHAVE_GRUPO_GLOBAL = "chave_escola_ano"
N_SPLITS_CV_GLOBAL = 3
N_ITER_BUSCA_GLOBAL = 10
LIMIAR_CLASSIFICACAO_GLOBAL = 0.50
REPETICOES_PERMUTACAO_GLOBAL = 5
REPETICOES_BOOTSTRAP_GLOBAL = 200

N_JOBS_MODELAGEM_GLOBAL = 1
N_SPLITS_HOLDOUT_GLOBAL = 1
SCORING_PRINCIPAL_GLOBAL = "roc_auc"
VERBOSE_BUSCA_GLOBAL = 2
ERRO_BUSCA_GLOBAL = "raise"
RETORNAR_SCORE_TREINO_GLOBAL = True

# Contrato comum de pre-processamento dos modelos.
IMPUTACAO_NUMERICA_GLOBAL = "median"
IMPUTACAO_CATEGORICA_GLOBAL = "most_frequent"
ONE_HOT_DESCONHECIDO_GLOBAL = "ignore"
ONE_HOT_SPARSO_GLOBAL = False  # matriz densa: evita fallback dos modelos para CPU
SPARSE_THRESHOLD_GLOBAL = 0.0
ESCALONADOR_CENTRALIZAR_GLOBAL = False


# Percentuais de amostragem (0 < valor <= 1). Alterar aqui afeta todo o notebook.
# O teste final permanece integral (100%); as amostras menores servem apenas ao
# desenvolvimento, comparacao de modelos e interpretabilidade.
FRACOES_AMOSTRAGEM = {
    "eda_correlacao": 0.05,
    "validacao_preparacao": 0.02,
    "fase5_modelos": 0.10,
    "fase5_knn": 0.02,  # KNN tem custo de memoria e predicao maior.
    "fase6_desenvolvimento": 0.12,
    "fase6_validacao_interna": 0.25,
    "fase7_desenvolvimento": 1.00,
    "fase7_teste": 1.00,
    "interpretacao": 0.035,
    "bootstrap": 0.17,
}

for _nome, _fracao in FRACOES_AMOSTRAGEM.items():
    if not 0 < _fracao <= 1:
        raise ValueError(f"Fracao invalida em FRACOES_AMOSTRAGEM[{_nome!r}]: {_fracao}")


def tamanho_por_fracao(total, fracao):
    """Converte uma proporcao em quantidade reproduzivel, sem exceder o total."""
    if total < 1:
        raise ValueError("Nao e possivel amostrar uma particao vazia.")
    return min(total, max(1, round(total * fracao)))


def materializar_split_percentual(dados, contrato, nome_split, fracao, seed, materializador):
    """Materializa uma fracao de um split e informa claramente o universo usado."""
    import polars as pl

    coluna_split = contrato["controle_split"]
    total = (
        dados.filter(pl.col(coluna_split) == nome_split)
        .select(pl.len().alias("total"))
        .collect()
        .item()
    )
    quantidade = tamanho_por_fracao(total, fracao)
    X, y = materializador(dados, contrato, nome_split, limit=quantidade, seed=seed)
    print(f"{nome_split}: {quantidade:,}/{total:,} registros ({fracao:.1%})")
    return X, y


def _estado_memoria_sistema():
    """Retorna RAM total e disponível em bytes."""
    valores = {}
    try:
        with open("/proc/meminfo", encoding="utf-8") as arquivo:
            for linha in arquivo:
                chave, valor = linha.split(":", 1)
                valores[chave] = int(valor.strip().split()[0]) * 1024
        return valores["MemTotal"], valores.get("MemAvailable", valores.get("MemFree", 0))
    except (OSError, KeyError, ValueError):
        try:
            return (
                os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
                os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
            )
        except (AttributeError, OSError, ValueError):
            return None, None


RAM_TOTAL_BYTES, RAM_DISPONIVEL_BYTES = _estado_memoria_sistema()
RAM_TOTAL_GB = RAM_TOTAL_BYTES / 1024**3 if RAM_TOTAL_BYTES else 0.0
CPUS_LOGICAS = os.cpu_count() or 1

if RAM_TOTAL_GB and RAM_TOTAL_GB <= 4.5:
    _perfil_detectado = "baixo"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 8.5:
    _perfil_detectado = "medio"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 16.5:
    _perfil_detectado = "normal"
else:
    _perfil_detectado = "alto"

_perfis = {
    "baixo": (60, 1),
    "medio": (70, min(2, CPUS_LOGICAS)),
    "normal": (75, min(4, CPUS_LOGICAS)),
    "alto": (80, min(8, max(1, CPUS_LOGICAS // 2))),
}
PERFIL_ATIVO = _perfil_detectado if PERFIL_RECURSOS == "auto" else PERFIL_RECURSOS.lower()
if PERFIL_ATIVO not in _perfis:
    raise ValueError("PERFIL_RECURSOS deve ser auto, baixo, medio, normal ou alto.")

_limite_perfil, _threads_perfil = _perfis[PERFIL_ATIVO]
LIMITE_RAM_ATIVO = _limite_perfil if LIMITE_RAM_PERCENTUAL is None else int(LIMITE_RAM_PERCENTUAL)
THREADS_MAXIMAS = _threads_perfil if MAX_THREADS is None else int(MAX_THREADS)
if not 50 <= LIMITE_RAM_ATIVO <= 90:
    raise ValueError("LIMITE_RAM_PERCENTUAL deve ficar entre 50 e 90.")
if THREADS_MAXIMAS < 1:
    raise ValueError("MAX_THREADS deve ser pelo menos 1.")
THREADS_MAXIMAS = min(THREADS_MAXIMAS, CPUS_LOGICAS)

# Definido antes de NumPy, Polars, scikit-learn e cuML serem importados.
for _variavel, _valor in {
    "OMP_NUM_THREADS": str(THREADS_MAXIMAS),
    "OPENBLAS_NUM_THREADS": str(THREADS_MAXIMAS),
    "MKL_NUM_THREADS": str(THREADS_MAXIMAS),
    "NUMEXPR_NUM_THREADS": str(THREADS_MAXIMAS),
    "VECLIB_MAXIMUM_THREADS": str(THREADS_MAXIMAS),
    "POLARS_MAX_THREADS": str(THREADS_MAXIMAS),
    "MALLOC_ARENA_MAX": "2",
}.items():
    os.environ[_variavel] = _valor

HISTORICO_RECURSOS = []
_estado_recursos = {"nivel": None, "threads": None, "contexto_threads": None}


def _aplicar_threads(limite):
    """Limita pools BLAS/OpenMP já carregados; Polars usa o limite inicial."""
    limite = max(1, min(int(limite), THREADS_MAXIMAS))
    if _estado_recursos["threads"] == limite:
        return
    anterior = _estado_recursos.get("contexto_threads")
    if anterior is not None:
        try:
            anterior.__exit__(None, None, None)
        except Exception:
            pass
    try:
        from threadpoolctl import threadpool_limits
        contexto = threadpool_limits(limits=limite)
        contexto.__enter__()
    except Exception:
        contexto = None
    _estado_recursos["contexto_threads"] = contexto
    _estado_recursos["threads"] = limite


def _liberar_memoria_global(agressiva=False):
    """Libera caches transitórios sem apagar variáveis usadas pelo fluxo."""
    shell = get_ipython()
    if shell is not None:
        for nome in ("_", "__", "___"):
            shell.user_ns[nome] = None
    gc.collect()
    modulo_cupy = sys.modules.get("cupy")
    if modulo_cupy is not None:
        try:
            modulo_cupy.get_default_memory_pool().free_all_blocks()
            modulo_cupy.get_default_pinned_memory_pool().free_all_blocks()
        except Exception:
            pass
    if agressiva:
        modulo_plt = sys.modules.get("matplotlib.pyplot")
        if modulo_plt is not None:
            try:
                modulo_plt.close("all")
            except Exception:
                pass
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except (OSError, AttributeError):
            pass


def _gerenciar_recursos(*_args, **_kwargs):
    """Reduz paralelismo progressivamente e nunca interrompe uma célula."""
    total, disponivel = _estado_memoria_sistema()
    if not total:
        return
    uso = 100.0 * (total - disponivel) / total
    margem = LIMITE_RAM_ATIVO - uso
    if margem > 15:
        nivel, threads, limpar = "normal", THREADS_MAXIMAS, False
    elif margem > 7:
        nivel, threads, limpar = "moderado", max(1, THREADS_MAXIMAS // 2), True
    else:
        nivel, threads, limpar = "conservador", 1, True

    _aplicar_threads(threads)
    if limpar:
        _liberar_memoria_global(agressiva=margem <= 0)

    if nivel != _estado_recursos["nivel"]:
        registro = {
            "instante_utc": datetime.now(timezone.utc).isoformat(),
            "nivel": nivel,
            "threads": threads,
            "ram_usada_percentual": round(uso, 1),
            "ram_disponivel_gb": round(disponivel / 1024**3, 2),
        }
        HISTORICO_RECURSOS.append(registro)
        print(
            f"[recursos] nível={nivel}; threads={threads}; "
            f"RAM usada={uso:.1f}%; livre={disponivel / 1024**3:.1f} GB."
        )
        _estado_recursos["nivel"] = nivel


ipython = get_ipython()
if ipython is not None:
    # Impede callbacks duplicados ao reexecutar a configuração.
    for _atributo in ("_tc3_memory_guard", "_tc3_resource_guard"):
        _guard_anterior = getattr(ipython, _atributo, None)
        if _guard_anterior is not None:
            for _evento, _funcao in _guard_anterior:
                try:
                    ipython.events.unregister(_evento, _funcao)
                except (ValueError, KeyError):
                    pass
    _callbacks_recursos = [
        ("pre_run_cell", _gerenciar_recursos),
        ("post_run_cell", _gerenciar_recursos),
    ]
    for _evento, _funcao in _callbacks_recursos:
        ipython.events.register(_evento, _funcao)
    ipython._tc3_resource_guard = _callbacks_recursos

_gerenciar_recursos()
print(
    "Configuração automática ativa: "
    f"perfil={PERFIL_ATIVO}; RAM={RAM_TOTAL_GB:.1f} GB; CPUs={CPUS_LOGICAS}; "
    f"limite preventivo={LIMITE_RAM_ATIVO}%; threads máximas={THREADS_MAXIMAS}."
)

# Backend GPU: ativado antes de qualquer importação do scikit-learn.
cuda_conda = os.path.join(sys.prefix, "targets", "x86_64-linux")
if os.path.exists(os.path.join(cuda_conda, "include", "cuda.h")):
    os.environ.setdefault("CUDA_PATH", cuda_conda)
if USAR_GPU:
    try:
        from cuml.accel import install
        install(log_level="info")
        import cupy as cp
        dispositivo = cp.cuda.runtime.getDeviceProperties(0)
        nome_gpu = dispositivo["name"].decode() if isinstance(dispositivo["name"], bytes) else str(dispositivo["name"])
        memoria_gpu_gb = cp.cuda.runtime.memGetInfo()[1] / 1024**3
        BACKEND_ML = f"RAPIDS cuML — {nome_gpu} ({memoria_gpu_gb:.1f} GB de VRAM)"
    except Exception as exc:
        raise RuntimeError("GPU/RAPIDS indisponível. Abra no WSL2 e selecione o kernel tc3-gpu.") from exc
else:
    BACKEND_ML = "scikit-learn CPU"
print("Backend de Machine Learning:", BACKEND_ML)

from pathlib import Path
import hashlib
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def localizar_raiz(inicio: Path) -> Path:
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "dados").is_dir() and (candidato / "notebooks").is_dir():
            return candidato
    raise FileNotFoundError("Execute este notebook dentro do repositório tech_challenge_3.")

PROJECT_ROOT = localizar_raiz(Path.cwd().resolve())
print("Raiz do projeto:", PROJECT_ROOT)


# Contrato único de diretórios. Fontes sempre são lidas da área imutável.
import tempfile

FONTES_ORIGINAIS_DIR = PROJECT_ROOT / "dados" / "fontes_originais"
_TEMP_BASE = PROJECT_ROOT / ".execucao_temporaria"
_TEMP_BASE.mkdir(parents=True, exist_ok=True)
if SALVAR_ARTEFATOS:
    RAIZ_ARTEFATOS = PROJECT_ROOT
    _DIRETORIO_TEMPORARIO = None
else:
    _DIRETORIO_TEMPORARIO = tempfile.TemporaryDirectory(prefix="tc3_", dir=_TEMP_BASE)
    RAIZ_ARTEFATOS = Path(_DIRETORIO_TEMPORARIO.name)

DADOS_PROCESSADOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_processados"
DADOS_ANALITICOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_analiticos"
RESULTADOS_DIR = RAIZ_ARTEFATOS / "resultados"
RESULTADOS_FIGURAS_DIR = RESULTADOS_DIR / "figuras"
RESULTADOS_TABELAS_DIR = RESULTADOS_DIR / "tabelas"
RESULTADOS_METRICAS_DIR = RESULTADOS_DIR / "metricas"
RESULTADOS_RELATORIOS_DIR = RESULTADOS_DIR / "relatorios"
MODELOS_DIR = RAIZ_ARTEFATOS / "modelos"

for _diretorio_saida in (
    DADOS_PROCESSADOS_DIR, DADOS_ANALITICOS_DIR, RESULTADOS_FIGURAS_DIR,
    RESULTADOS_TABELAS_DIR, RESULTADOS_METRICAS_DIR,
    RESULTADOS_RELATORIOS_DIR, MODELOS_DIR,
):
    _diretorio_saida.mkdir(parents=True, exist_ok=True)

def deve_recalcular(caminho):
    """True quando o cache deve ser ignorado ou ainda não existe."""
    return not REUTILIZAR_ARTEFATOS or not Path(caminho).exists()


def publicar_parquet_particionado(frame, destino, *, particionar_por):
    """Publica partições sequencialmente e troca o diretório de forma atômica.

    O ``write_parquet(partition_by=...)`` do Polars prepara várias partições em
    paralelo e pode duplicar o pico de RAM de bases já materializadas. Esta
    implementação mantém apenas uma partição adicional em memória por vez.
    """
    import gc
    import shutil

    destino = Path(destino)
    temporario = destino.with_name(destino.name + "__publicacao_tmp")
    anterior = destino.with_name(destino.name + "__publicacao_anterior")
    for caminho in (temporario, anterior):
        if caminho.exists():
            shutil.rmtree(caminho)
    temporario.mkdir(parents=True, exist_ok=True)

    lazy = frame if isinstance(frame, pl.LazyFrame) else frame.lazy()
    schema = lazy.collect_schema()
    if particionar_por not in schema.names():
        raise ValueError(f"Coluna de particionamento ausente: {particionar_por}")

    valores = (
        lazy.select(pl.col(particionar_por).unique())
        .collect(engine="streaming")
        .get_column(particionar_por)
        .to_list()
    )
    valores = sorted(valores, key=lambda valor: (valor is None, str(valor)))
    if not valores:
        raise RuntimeError("Não há valores para particionar a publicação.")

    arquivos_novos = []
    for indice, valor in enumerate(valores):
        valor_hive = "__HIVE_DEFAULT_PARTITION__" if valor is None else str(valor)
        if any(separador in valor_hive for separador in ("/", "\\")):
            raise ValueError(f"Valor de partição inválido: {valor_hive!r}")
        diretorio_particao = temporario / f"{particionar_por}={valor_hive}"
        diretorio_particao.mkdir(parents=True, exist_ok=True)
        arquivo = diretorio_particao / f"part-{indice:05d}.parquet"
        predicado = pl.col(particionar_por).is_null() if valor is None else pl.col(particionar_por) == pl.lit(valor)
        lazy.filter(predicado).sink_parquet(
            arquivo,
            compression="zstd",
            statistics=True,
            row_group_size=100_000,
            mkdir=True,
            engine="streaming",
        )
        if not arquivo.is_file() or arquivo.stat().st_size == 0:
            raise RuntimeError(f"Partição não foi gravada corretamente: {arquivo}")
        arquivos_novos.append(arquivo)
        gc.collect()

    if destino.exists():
        destino.replace(anterior)
    try:
        try:
            temporario.replace(destino)
        except OSError:
            # Volumes Docker montados do Windows podem bloquear rename de diretórios.
            # O fallback copia a publicação pronta e remove a área temporária.
            shutil.copytree(temporario, destino)
            shutil.rmtree(temporario)
    except Exception:
        if destino.exists():
            shutil.rmtree(destino)
        if anterior.exists():
            anterior.replace(destino)
        raise
    if anterior.exists():
        shutil.rmtree(anterior)
    return len(arquivos_novos)


def salvar_figura_eda(figura, nome):
    """Exporta uma figura exploratória sem alterar sua exibição no notebook."""
    destino = RESULTADOS_FIGURAS_DIR / "fluxo_exploratorio" / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    figura.savefig(destino, dpi=180, bbox_inches="tight")
    return destino

print("Fontes originais:", FONTES_ORIGINAIS_DIR)
print("Destino dos artefatos:", RAIZ_ARTEFATOS)
print("Reutilizar artefatos:", REUTILIZAR_ARTEFATOS, "| Salvar:", SALVAR_ARTEFATOS)

# Imports compartilhados que, no fluxo completo, são carregados por etapas anteriores.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


[recursos] nível=normal; threads=4; RAM usada=34.1%; livre=10.2 GB.
Configuração automática ativa: perfil=normal; RAM=15.5 GB; CPUs=20; limite preventivo=75%; threads máximas=4.
[cuml.accel] Accelerator installed.
Backend de Machine Learning: RAPIDS cuML — NVIDIA GeForce RTX 3050 6GB Laptop GPU (6.0 GB de VRAM)
Raiz do projeto: /workspace
Fontes originais: /workspace/dados/fontes_originais
Destino dos artefatos: /workspace
Reutilizar artefatos: False | Salvar: True


In [2]:
MODELOS_GPU_OBRIGATORIOS = {"LogisticRegression", "KNeighborsClassifier", "LinearSVC", "RandomForestClassifier"}
def validar_backend_modelo(modelo):
    estimador = modelo.named_steps.get("model", modelo) if hasattr(modelo, "named_steps") else modelo
    nome = type(estimador).__name__
    executou_gpu = getattr(estimador, "_gpu", None) is not None
    if USAR_GPU:
        if nome in MODELOS_GPU_OBRIGATORIOS and not executou_gpu:
            raise RuntimeError(f"{nome} não executou na GPU. Interrompa: não publique este resultado como GPU.")
        backend = "GPU" if executou_gpu else "CPU esperada"
    else:
        backend = "CPU"
    print(f"Backend validado — {nome}: {backend}")


In [3]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import polars as pl
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, RandomizedSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEED = SEED_GLOBAL
FRACAO_DESENVOLVIMENTO = FRACOES_AMOSTRAGEM["fase6_desenvolvimento"]
FRACAO_VALIDACAO = FRACOES_AMOSTRAGEM["fase6_validacao_interna"]
N_SPLITS_CV = N_SPLITS_CV_GLOBAL
N_ITER_BUSCA = N_ITER_BUSCA_GLOBAL
CHAVE_GRUPO = CHAVE_GRUPO_GLOBAL

# PROJECT_ROOT e herdado da configuracao central.
MANIFEST_PATH = DADOS_PROCESSADOS_DIR / "preparado" / "manifesto_base_modelagem_com_chave_escola.json"
RESULT_DIR = RESULTADOS_DIR / "metricas" / "fase_6"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

contrato = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
FEATURES = contrato["features_modelagem"]
TARGET = contrato["target"]
SPLIT = contrato["controle_split"]

assert CHAVE_GRUPO in contrato["chaves_rastreabilidade"]
assert CHAVE_GRUPO not in FEATURES, "A chave de escola não pode entrar como feature."
print("Projeto:", PROJECT_ROOT)
print("Features:", len(FEATURES), "| agrupamento:", CHAVE_GRUPO)
print("Teste reservado: sim")


Projeto: /workspace
Features: 18 | agrupamento: chave_escola_ano
Teste reservado: sim


## 2. Leitura e separação agrupada dos dados


In [4]:
data_glob = PROJECT_ROOT / Path(contrato["padrao_leitura"])
lf = pl.scan_parquet(str(data_glob), hive_partitioning=True)
schema = lf.collect_schema()
obrigatorias = set(FEATURES + [TARGET, SPLIT, "id_aluno", CHAVE_GRUPO])
ausentes = sorted(obrigatorias - set(schema.names()))
if ausentes:
    raise KeyError(f"Colunas obrigatórias ausentes: {ausentes}")

# O teste nao e lido: treino e validacao formam o universo de desenvolvimento.
lf_desenvolvimento = lf.filter(pl.col(SPLIT).is_in(["treino", "validacao"]))
total_desenvolvimento = lf_desenvolvimento.select(pl.len()).collect().item()
quantidade_desenvolvimento = tamanho_por_fracao(
    total_desenvolvimento, FRACAO_DESENVOLVIMENTO
)
desenvolvimento = (
    lf_desenvolvimento
    .with_columns(pl.col("id_aluno").hash(seed=SEED).alias("_ordem_amostra"))
    .sort("_ordem_amostra")
    .head(quantidade_desenvolvimento)
    .select(FEATURES + [TARGET, CHAVE_GRUPO])
    .collect()
)
print(
    f"Fase 6 — Regressao Logistica: {quantidade_desenvolvimento:,}/"
    f"{total_desenvolvimento:,} registros de desenvolvimento "
    f"({FRACAO_DESENVOLVIMENTO:.1%})."
)
if desenvolvimento.is_empty():
    raise ValueError("A amostra de desenvolvimento ficou vazia.")

grupos = desenvolvimento[CHAVE_GRUPO].cast(pl.String).to_numpy()
separador = GroupShuffleSplit(
    n_splits=N_SPLITS_HOLDOUT_GLOBAL,
    test_size=FRACAO_VALIDACAO,
    random_state=SEED,
)
idx_treino, idx_validacao = next(
    separador.split(
        desenvolvimento.select(FEATURES),
        desenvolvimento[TARGET].to_numpy(),
        groups=grupos,
    )
)

X_train = desenvolvimento[idx_treino.tolist()].select(FEATURES)
y_train = desenvolvimento[idx_treino.tolist()][TARGET].to_numpy()
g_train = grupos[idx_treino]
X_valid = desenvolvimento[idx_validacao.tolist()].select(FEATURES)
y_valid = desenvolvimento[idx_validacao.tolist()][TARGET].to_numpy()
g_valid = grupos[idx_validacao]

sobreposicao = set(g_train.tolist()) & set(g_valid.tolist())
assert not sobreposicao, f"Há {len(sobreposicao)} grupos repetidos entre treino e validação."
print({
    "treino": len(y_train), "validacao": len(y_valid),
    "grupos_treino": int(np.unique(g_train).size),
    "grupos_validacao": int(np.unique(g_valid).size),
    "sobreposicao_grupos": len(sobreposicao),
    "prevalencia_treino": float(np.mean(y_train)),
    "prevalencia_validacao": float(np.mean(y_valid)),
})


Fase 6 — Regressao Logistica: 394,366/3,286,383 registros de desenvolvimento (12.0%).
{'treino': 295312, 'validacao': 99054, 'grupos_treino': 55506, 'grupos_validacao': 18503, 'sobreposicao_grupos': 0, 'prevalencia_treino': 0.5136465839518882, 'prevalencia_validacao': 0.5117612615341127}


## 3. Funções e métricas comuns


In [5]:
CATEGORICAS = ["sigla_uf", "rede"]
NUMERICAS = [c for c in FEATURES if c not in CATEGORICAS]
# Seletores posicionais evitam a chamada pandas-específica Index.get_loc()
# do ColumnTransformer quando X é um DataFrame Polars.
INDICES_CATEGORICAS = [FEATURES.index(c) for c in CATEGORICAS]
INDICES_NUMERICAS = [FEATURES.index(c) for c in NUMERICAS]

def criar_preprocessador(tipo: str) -> ColumnTransformer:
    if tipo == "linear":
        cat = Pipeline([
            ("imputer", SimpleImputer(strategy=IMPUTACAO_CATEGORICA_GLOBAL)),
            ("encoder", OneHotEncoder(handle_unknown=ONE_HOT_DESCONHECIDO_GLOBAL, sparse_output=ONE_HOT_SPARSO_GLOBAL, dtype=np.float32)),
        ])
        num = Pipeline([
            ("imputer", SimpleImputer(strategy=IMPUTACAO_NUMERICA_GLOBAL)),
            ("scaler", StandardScaler(with_mean=ESCALONADOR_CENTRALIZAR_GLOBAL)),
        ])
        return ColumnTransformer(
            [("categoricas", cat, INDICES_CATEGORICAS), ("numericas", num, INDICES_NUMERICAS)],
            sparse_threshold=SPARSE_THRESHOLD_GLOBAL,  # força matriz densa para o cuML
        )
    if tipo == "tree":
        cat = Pipeline([
            ("imputer", SimpleImputer(strategy=IMPUTACAO_CATEGORICA_GLOBAL)),
            ("encoder", OneHotEncoder(handle_unknown=ONE_HOT_DESCONHECIDO_GLOBAL, sparse_output=ONE_HOT_SPARSO_GLOBAL, dtype=np.float32)),
        ])
        num = Pipeline([("imputer", SimpleImputer(strategy=IMPUTACAO_NUMERICA_GLOBAL))])
        return ColumnTransformer(
            [("categoricas", cat, INDICES_CATEGORICAS), ("numericas", num, INDICES_NUMERICAS)],
            sparse_threshold=SPARSE_THRESHOLD_GLOBAL,  # força matriz densa para o cuML
        )
    if tipo == "hist":
        cat = Pipeline([
            ("imputer", SimpleImputer(strategy=IMPUTACAO_CATEGORICA_GLOBAL)),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ])
        num = Pipeline([("imputer", SimpleImputer(strategy=IMPUTACAO_NUMERICA_GLOBAL))])
        return ColumnTransformer(
            [("categoricas", cat, INDICES_CATEGORICAS), ("numericas", num, INDICES_NUMERICAS)],
            sparse_threshold=SPARSE_THRESHOLD_GLOBAL,
        )
    raise ValueError(f"Preprocessador desconhecido: {tipo}")

def pontuacao(modelo, X):
    if hasattr(modelo, "predict_proba"):
        return np.asarray(modelo.predict_proba(X))[:, 1]
    return np.asarray(modelo.decision_function(X))

def metricas(y_true, y_pred, y_score):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "average_precision": float(average_precision_score(y_true, y_score)),
        "matriz_confusao": confusion_matrix(y_true, y_pred).tolist(),
    }

SCORING = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
}
cv_agrupada = StratifiedGroupKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=SEED)


## 4. Busca moderada — Regressão Logística

A busca é deliberadamente limitada a **10 configurações × 3 folds agrupados**. Isso representa evolução comparável dos candidatos, sem transformar esta etapa em tuning exaustivo.


In [6]:
from sklearn.linear_model import LogisticRegression

# `lbfgs` é determinístico neste pipeline. A tolerância 1e-3 evita exigir
# precisão numérica sem ganho preditivo material; a convergência vencedora é
# verificada explicitamente antes da exportação.
estimador = LogisticRegression(
    random_state=SEED,
    solver="lbfgs",
    max_iter=10_000,
    tol=1e-3,
)
pipeline = Pipeline([
    ("preprocessor", criar_preprocessador("linear")),
    ("model", estimador),
])
espaco_busca = {
    "model__C": np.logspace(-3, 1, 20),
    "model__class_weight": [None, "balanced"],
}

busca = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=espaco_busca,
    n_iter=N_ITER_BUSCA,
    scoring=SCORING,
    refit=SCORING_PRINCIPAL_GLOBAL,
    cv=cv_agrupada,
    random_state=SEED,
    n_jobs=N_JOBS_MODELAGEM_GLOBAL,  # uma busca por vez para não disputar a GPU
    verbose=VERBOSE_BUSCA_GLOBAL,
    return_train_score=RETORNAR_SCORE_TREINO_GLOBAL,
    error_score=ERRO_BUSCA_GLOBAL,
)

inicio = time.perf_counter()
with warnings.catch_warnings(record=True) as avisos_busca:
    warnings.simplefilter("always")
    busca.fit(X_train, y_train, groups=g_train)
    validar_backend_modelo(busca.best_estimator_)
tempo_busca = time.perf_counter() - inicio
avisos_convergencia_busca = [
    aviso for aviso in avisos_busca if issubclass(aviso.category, ConvergenceWarning)
]

print("Melhores parâmetros:", busca.best_params_)
print("ROC AUC médio na CV agrupada:", busca.best_score_)
print("Tempo total da busca (s):", tempo_busca)
print("Avisos de convergência durante a busca:", len(avisos_convergencia_busca))


Fitting 3 folds for each of 10 candidates, totalling 30 fits
[cuml.accel] `LogisticRegression.fit` ran on GPU
[cuml.accel] `LogisticRegression.decision_function` ran on GPU
[cuml.accel] `LogisticRegression.predict` ran on GPU
[cuml.accel] `LogisticRegression.decision_function` ran on GPU
[cuml.accel] `LogisticRegression.predict` ran on GPU
[CV] END model__C=0.07847599703514611, model__class_weight=balanced; total time=   1.3s
[cuml.accel] `LogisticRegression.fit` ran on GPU
[cuml.accel] `LogisticRegression.decision_function` ran on GPU
[cuml.accel] `LogisticRegression.predict` ran on GPU
[cuml.accel] `LogisticRegression.decision_function` ran on GPU
[cuml.accel] `LogisticRegression.predict` ran on GPU
[CV] END model__C=0.07847599703514611, model__class_weight=balanced; total time=   1.1s
[cuml.accel] `LogisticRegression.fit` ran on GPU
[cuml.accel] `LogisticRegression.decision_function` ran on GPU
[cuml.accel] `LogisticRegression.predict` ran on GPU
[cuml.accel] `LogisticRegression.dec

## 5. Avaliação na validação agrupada externa


In [7]:
melhor_modelo = busca.best_estimator_
estimador_vencedor = melhor_modelo.named_steps["model"]
iteracoes_vencedor = int(np.max(estimador_vencedor.n_iter_))
convergiu_vencedor = iteracoes_vencedor < estimador_vencedor.max_iter
if not convergiu_vencedor:
    raise RuntimeError(
        "A configuração logística vencedora não convergiu; não exporte nem compare este resultado."
    )
print("Convergência da configuração vencedora: sim | iterações:", iteracoes_vencedor)

pred_treino = melhor_modelo.predict(X_train)
score_treino = pontuacao(melhor_modelo, X_train)
pred_valid = melhor_modelo.predict(X_valid)
score_valid = pontuacao(melhor_modelo, X_valid)

metricas_treino = metricas(y_train, pred_treino, score_treino)
metricas_validacao = metricas(y_valid, pred_valid, score_valid)
gap_auc = metricas_treino["roc_auc"] - metricas_validacao["roc_auc"]

display(pl.DataFrame([
    {"conjunto": "treino", **{k: v for k, v in metricas_treino.items() if k != "matriz_confusao"}},
    {"conjunto": "validacao_agrupada", **{k: v for k, v in metricas_validacao.items() if k != "matriz_confusao"}},
]))
print("Gap ROC AUC treino-validação:", gap_auc)
print("Matriz de confusão da validação:", metricas_validacao["matriz_confusao"])

colunas_cv = [
    "rank_test_roc_auc", "mean_test_roc_auc", "std_test_roc_auc",
    "mean_test_average_precision", "mean_test_balanced_accuracy", "mean_test_f1",
    "mean_train_roc_auc", "mean_fit_time", "params",
]
dados_cv = {}
for coluna in colunas_cv:
    valores = busca.cv_results_[coluna]
    if coluna == "params":
        dados_cv[coluna] = [json.dumps(item, ensure_ascii=False, sort_keys=True, default=str) for item in valores]
    else:
        if hasattr(valores, "filled"):
            valores = valores.filled(np.nan)
        dados_cv[coluna] = valores.tolist() if hasattr(valores, "tolist") else list(valores)
cv_resultados = pl.DataFrame(dados_cv, strict=False).sort("rank_test_roc_auc")
display(cv_resultados.select(colunas_cv).head(10))


[recursos] nível=normal; threads=4; RAM usada=58.2%; livre=6.5 GB.
Convergência da configuração vencedora: sim | iterações: 78
[cuml.accel] `LogisticRegression.predict` ran on GPU
[cuml.accel] `LogisticRegression.predict_proba` ran on GPU
[cuml.accel] `LogisticRegression.predict` ran on GPU
[cuml.accel] `LogisticRegression.predict_proba` ran on GPU


conjunto,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
str,f64,f64,f64,f64,f64,f64,f64
"""treino""",0.563438,0.562922,0.574033,0.581814,0.577897,0.594061,0.603545
"""validacao_agrupada""",0.561502,0.561083,0.570551,0.578868,0.57468,0.59206,0.599088


Gap ROC AUC treino-validação: 0.002001490880956802
Matriz de confusão da validação: [[26275, 22087], [21348, 29344]]


rank_test_roc_auc,mean_test_roc_auc,std_test_roc_auc,mean_test_average_precision,mean_test_balanced_accuracy,mean_test_f1,mean_train_roc_auc,mean_fit_time,params
i64,f64,f64,f64,f64,f64,f64,f64,str
1,0.605605,0.010646,0.617969,0.572612,0.594469,0.605589,1.459732,"""{""model__C"": 0.004281332398719…"
2,0.603045,0.011645,0.613311,0.57053,0.588982,0.603726,1.809223,"""{""model__C"": 0.002636650898730…"
3,0.596186,0.019624,0.606408,0.567509,0.588841,0.597393,2.178293,"""{""model__C"": 0.545559478116851…"
4,0.589049,0.010421,0.596721,0.560926,0.580674,0.58981,1.500318,"""{""model__C"": 0.018329807108324…"
5,0.588899,0.010324,0.597134,0.561632,0.581988,0.589761,0.794183,"""{""model__C"": 0.048329302385717…"
6,0.58841,0.008403,0.596018,0.563454,0.556064,0.588072,1.131355,"""{""model__C"": 0.029763514416313…"
7,0.588376,0.008408,0.596025,0.564426,0.554856,0.588089,1.298267,"""{""model__C"": 0.545559478116851…"
8,0.586468,0.01486,0.592641,0.56389,0.5632,0.587652,0.933951,"""{""model__C"": 0.078475997035146…"
9,0.5822,0.007938,0.588324,0.560651,0.555755,0.582209,1.579978,"""{""model__C"": 6.158482110660261…"


[recursos] nível=moderado; threads=2; RAM usada=60.3%; livre=6.1 GB.


### Nota de reprodutibilidade e comparação com o fluxo oficial

Nesta execução isolada, a busca selecionou `C = 0.004281332398719396`, com ROC AUC médio de validação cruzada de aproximadamente `0,605605`. No notebook consolidado `TC3_Pipeline_Completo_Alfabetizacao.ipynb`, a configuração vencedora havia sido `C = 0.0026366508987303583`, com ROC AUC médio de aproximadamente `0,612312`.

Essa diferença não representa mudança no conjunto de dados, no protocolo de validação ou no objetivo da análise. As duas configurações aplicam regularização forte e produziram desempenho praticamente equivalente no holdout de validação externa:

- notebook consolidado: ROC AUC de aproximadamente `0,591683`;
- execução isolada: ROC AUC de aproximadamente `0,592060`.

Mesmo com sementes fixas, o backend acelerado por GPU pode apresentar pequenas variações relacionadas à ordem das operações paralelas, à precisão numérica e ao processo de convergência. Neste caso, a variação foi suficiente para alterar o melhor valor de `C` e a média interna da validação cruzada, mas não alterou a avaliação externa nem a posição da Regressão Logística no ranking.

Para manter o particionamento como representação modular do fluxo oficial, a referência metodológica permanece a decisão registrada em `TC3_Pipeline_Completo_Alfabetizacao.ipynb`. A execução isolada é preservada como evidência complementar e não redefine a seleção final.

Em ambas as execuções, a Regressão Logística permaneceu abaixo do Random Forest e do Gradient Boosting. Portanto, a divergência não altera a escolha oficial do Random Forest nem a avaliação final no conjunto de teste.

## 6. Exportação padronizada para a seleção dos modelos


In [8]:
def tornar_json(valor):
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating,)):
        return float(valor)
    if isinstance(valor, np.ndarray):
        return valor.tolist()
    raise TypeError(f"Tipo não serializável: {type(valor).__name__}")

melhor_indice = int(busca.best_index_)
resultado = {
    "schema_resultado": 1,
    "fase": 6,
    "modelo": {"id": "TC3-60", "nome": "Regressão Logística"},
    "protocolo": {
        "manifesto": MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "features": FEATURES,
        "chave_agrupamento": CHAVE_GRUPO,
        "semantica_agrupamento": contrato.get("semantica_chave_escola"),
        "amostra_desenvolvimento": int(desenvolvimento.height),
        "registros_treino": int(len(y_train)),
        "registros_validacao": int(len(y_valid)),
        "grupos_treino": int(np.unique(g_train).size),
        "grupos_validacao": int(np.unique(g_valid).size),
        "sobreposicao_grupos": int(len(sobreposicao)),
        "cv": "StratifiedGroupKFold",
        "n_splits_cv": N_SPLITS_CV,
        "seed": SEED,
        "teste_utilizado": False,
    },
    "busca": {
        "metodo": "RandomizedSearchCV",
        "n_iter": N_ITER_BUSCA,
        "metrica_selecao": "roc_auc",
        "melhores_parametros": busca.best_params_,
        "roc_auc_cv_media": float(busca.best_score_),
        "roc_auc_cv_desvio": float(busca.cv_results_["std_test_roc_auc"][melhor_indice]),
        "tempo_total_segundos": float(tempo_busca),
        "convergiu_vencedor": bool(convergiu_vencedor),
        "iteracoes_vencedor": int(iteracoes_vencedor),
        "avisos_convergencia_busca": len(avisos_convergencia_busca),
        "convergiu_vencedor": bool(convergiu_vencedor),
        "iteracoes_vencedor": int(iteracoes_vencedor),
        "avisos_convergencia_busca": len(avisos_convergencia_busca),
    },
    "metricas_treino": metricas_treino,
    "metricas_validacao": metricas_validacao,
    "gap_roc_auc": float(gap_auc),
    "top_resultados_cv": cv_resultados.select(colunas_cv).head(10).to_dicts(),
    "observacoes": [
        "id_escola e chave_escola_ano foram usados somente para agrupamento/rastreabilidade.",
        "O conjunto de teste não foi carregado nem utilizado na seleção.",
        "A chave de escola é anônima e anual; não identifica a escola real nem conecta anos diferentes.",
        "São Paulo possui dados somente em 2024.",
    ],
}

OUTPUT_PATH = RESULT_DIR / "TC3-60_resultado_otimizacao.json"
OUTPUT_PATH.write_text(
    json.dumps(resultado, ensure_ascii=False, indent=2, default=tornar_json),
    encoding="utf-8",
)
print("Resultado salvo em:", OUTPUT_PATH)


[recursos] nível=normal; threads=4; RAM usada=59.2%; livre=6.3 GB.
Resultado salvo em: /workspace/resultados/metricas/fase_6/TC3-60_resultado_otimizacao.json


## 7. Como interpretar

O resultado deste notebook ainda **não é a avaliação final**. Compare-o aos outros candidatos no notebook TC3-63. O vencedor será escolhido antes de qualquer uso do teste. Convergência, estabilidade entre folds, gap e custo computacional devem ser discutidos junto com as métricas.
